# Single-Unit Evaluation Notebook

## Purpose

This notebook isolates each generation asset (Heat Pump, Condensing Boiler, CHP) and evaluates it standalone using two optimization frameworks:

-  Standard 24h MILP (myopic, no look-ahead)
-  MPC-style 35h solve / 24h commit (look-ahead)

This yields 6 configurations total (3 units × 2 approaches), all evaluated under the same conditions.

## Problem Scope

For each single-unit run, all other generation assets are disabled (their dispatch variables are fixed to zero). The thermal storage remains active in all configurations to allow feasibility across all demand profiles. The heat balance constraint must still be satisfied at every timestep.

## Reference

Full system MILP formulation: `docs/optimization/optimization_problem.md`

---
## Shared Setup

All imports, parameters, data loading, and helper functions shared across all approach cells.

In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass, field
from time import perf_counter
import pyomo.environ as pyo

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 120


NOTEBOOK_DIR = Path().resolve()

BASE_DIR = next(p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
                if (p / "data").exists())

DATA_DIR = BASE_DIR / "data"

DEMAND_FILE = DATA_DIR / "RawData_MeasuredHeadDemand.csv"
PRICE_FILE = DATA_DIR / "Gro_handelspreise_202403010000_202404020000_Stunde.csv"

In [40]:
@dataclass
class ModelParams:
    # Time settings
    dt: float = 0.25
    n_hours: int = 24

    # Prices and emissions
    gas_price: float = 35.0
    co2_factor: float = 0.201
    co2_price: float = 60.0

    # Storage
    sto_capacity: float = 200.0
    sto_charge_max: float = 15.0
    sto_discharge_max: float = 15.0
    sto_loss: float = 0.000125
    sto_soc_init: float = 200.0

    # Heat pump
    hp_p_el_min: float = 1.0
    hp_p_el_max: float = 8.0
    hp_cop: float = 3.5

    # Boiler
    boiler_q_min: float = 2.0
    boiler_q_max: float = 12.0
    boiler_eff: float = 0.97
    boiler_min_up: int = 4
    boiler_min_down: int = 4

    # CHP
    chp_p_el_min: float = 2.0
    chp_p_el_max: float = 6.0
    chp_eta_el: float = 0.40
    chp_eta_th: float = 0.48
    chp_min_up: int = 8
    chp_min_down: int = 8
    chp_startup_cost: float = 600.0

    # Terminal storage handling
    soc_target: float = 100.0
    lambda_term: float = 20.0
    use_hard_soc_min: bool = True
    soc_min: float = 50.0

    # Initial states
    hp_on_0: int = 0
    boiler_on_0: int = 0
    chp_on_0: int = 0
    soc_0: float = field(default=200.0)
    boiler_time_in_state_0: int = 0
    chp_time_in_state_0: int = 0

    #penalty
    slack_penalty: float = 1000

    # ---- Derived properties ----
    @property
    def fuel_cost(self):
        return self.gas_price + self.co2_factor * self.co2_price

    @property
    def n_steps(self):
        return int(self.n_hours / self.dt)

    @property
    def time_index(self):
        return range(1, self.n_steps + 1)

    @property
    def chp_th_per_el(self):
        return self.chp_eta_th / self.chp_eta_el

    @property
    def chp_q_th_min(self):
        return self.chp_th_per_el * self.chp_p_el_min

    @property
    def chp_q_th_max(self):
        return self.chp_th_per_el * self.chp_p_el_max

In [41]:
def load_heat_demand(demand_file: Path) -> pd.DataFrame:
    df = pd.read_csv(demand_file)
    df["timestamp"] = pd.to_datetime(df["Time Point"], utc=True)
    df["demand_mw_th"] = df["Measured Heat Demand[W]"] / 1e6
    
    return df[["timestamp", "demand_mw_th"]].set_index("timestamp").sort_index()

def load_electricity_prices(price_file: Path) -> pd.DataFrame:
    df = pd.read_csv(price_file, sep=";", decimal=",", low_memory=False)
    price_col = next(
        c for c in df.columns if c.startswith("Deutschland/Luxemburg [€/MWh]")
    )
    ts = pd.to_datetime(df["Datum von"], format="%d.%m.%Y %H:%M")
    ts = ts.dt.tz_localize("Europe/Berlin", ambiguous="infer", nonexistent="shift_forward")
    df["timestamp"] = ts.dt.tz_convert("UTC")
    df["price_eur_mwh"] = df[price_col]

    return df[["timestamp", "price_eur_mwh"]].set_index("timestamp").sort_index()


params = ModelParams()
heat_demand_raw = load_heat_demand(DEMAND_FILE)
electricity_prices_raw = load_electricity_prices(PRICE_FILE)

COMMON_HOURLY_INDEX = heat_demand_raw.index.intersection(electricity_prices_raw.index).sort_values()
EVALUATION_START = COMMON_HOURLY_INDEX[0]
EVALUATION_DAYS = 67
COMMIT_HOURS = 24
COMMIT_STEPS = int(COMMIT_HOURS / params.dt)

RESULT_COLUMNS = ["approach_name", "unit", "runtime_seconds", "notes", "total_cost_eur", "profile"]
RESULTS = pd.DataFrame(columns=RESULT_COLUMNS)

#print(f"Project root : {PROJECT_ROOT}")
print(f"Evaluation start : {EVALUATION_START}")
print(f"Evaluation days  : {EVALUATION_DAYS}")
print(f"Commit steps/day : {COMMIT_STEPS}")
print(f"Effective gas cost: {params.fuel_cost:.2f} EUR/MWh_Hs")

Evaluation start : 2024-02-29 23:00:00+00:00
Evaluation days  : 67
Commit steps/day : 96
Effective gas cost: 47.06 EUR/MWh_Hs


In [42]:
# ---------------------------------------------------------------------------
# Shared helper functions
# ---------------------------------------------------------------------------

def prepare_hourly_window(start_timestamp, n_hours, demand_raw, price_raw):
    start_timestamp = pd.Timestamp(start_timestamp)
    if start_timestamp.tzinfo is None:
        start_timestamp = start_timestamp.tz_localize('UTC')
    else:
        start_timestamp = start_timestamp.tz_convert('UTC')
    hourly_index = pd.date_range(start=start_timestamp, periods=n_hours, freq='1h', tz='UTC')
    demand_hourly = demand_raw.reindex(hourly_index)['demand_mw_th']
    price_hourly = price_raw.reindex(hourly_index)['price_eur_mwh']
    demand_hourly = demand_hourly.ffill().bfill()
    price_hourly = price_hourly.ffill().bfill()
    timestamps = pd.date_range(
        start=hourly_index[0],
        periods=n_hours * int(1 / params.dt),
        freq=pd.to_timedelta(params.dt, unit='h'),
    )
    return (
        np.repeat(demand_hourly.to_numpy(dtype=float), int(1 / params.dt)),
        np.repeat(price_hourly.to_numpy(dtype=float), int(1 / params.dt)),
        timestamps,
    )


def build_result_profile(model, demand_th, price_el, timestamps) -> pd.DataFrame:
    dispatch = pd.DataFrame(
        {
            "timestamp": pd.to_datetime(timestamps, utc=True),
            "demand_th": np.asarray(demand_th, dtype=float),
            "price_el": np.asarray(price_el, dtype=float),
            "hp_th_out": [pyo.value(model.hp_th_out[t]) for t in model.T],
            "boiler_th_out": [pyo.value(model.boiler_th_out[t]) for t in model.T],
            "chp_th_out": [pyo.value(model.chp_th_out[t]) for t in model.T],
            "sto_th_discharge": [pyo.value(model.sto_th_discharge[t]) for t in model.T],
            "sto_th_charge": [pyo.value(model.sto_th_charge[t]) for t in model.T],
            "hp_el_in": [pyo.value(model.hp_el_in[t]) for t in model.T],
            "chp_el_out": [pyo.value(model.chp_el_out[t]) for t in model.T],
            "sto_th_soc": [pyo.value(model.sto_th_soc[t]) for t in model.T],
            "heat_slack": [pyo.value(model.heat_slack[t]) for t in model.T],
        }
    )
    dispatch.index.name = "step"
    return dispatch


def compute_committed_cost(model, run_params, commit_steps) -> float:
    return float(
        sum(
            pyo.value(model.hp_el_in[t]) * pyo.value(model.price_el[t]) * run_params.dt
            + pyo.value(model.boiler_fuel_in[t]) * run_params.fuel_cost * run_params.dt
            + pyo.value(model.chp_fuel_in[t]) * run_params.fuel_cost * run_params.dt
            + run_params.chp_startup_cost * pyo.value(model.chp_start[t])
            - pyo.value(model.chp_el_out[t]) * pyo.value(model.price_el[t]) * run_params.dt
            #Use the same price as effictive gas cost for slack heat to keep its cost realistic
            + 47.06 * pyo.value(model.heat_slack[t]) * run_params.dt
            for t in range(commit_steps)
        )
    )


def extract_commit_state(model, run_params, commit_steps) -> dict:
    last_t = commit_steps - 1

    def binary_value(var, idx):
        return int(round(pyo.value(var[idx])))

    def consecutive_state_length(var, initial_value, initial_length):
        final_value = binary_value(var, last_t)
        count = 0
        for t in range(last_t, -1, -1):
            if binary_value(var, t) == final_value:
                count += 1
            else:
                break
        if count == commit_steps and initial_value == final_value:
            return final_value, initial_length + count
        return final_value, count

    boiler_on_0, boiler_time_in_state_0 = consecutive_state_length(
        model.boiler_on, run_params.boiler_on_0, run_params.boiler_time_in_state_0,
    )
    chp_on_0, chp_time_in_state_0 = consecutive_state_length(
        model.chp_on, run_params.chp_on_0, run_params.chp_time_in_state_0,
    )

    return {
        'hp_on_0': binary_value(model.hp_on, last_t),
        'boiler_on_0': boiler_on_0,
        'chp_on_0': chp_on_0,
        'soc_0': float(pyo.value(model.sto_th_soc[last_t])),
        'boiler_time_in_state_0': boiler_time_in_state_0,
        'chp_time_in_state_0': chp_time_in_state_0,
    }


def add_min_up_down_constraints(model, on_var, start_var, stop_var, min_up, min_down, prefix):
    if min_up > 0:
        def min_up_rule(m, t):
            start_idx = max(0, t - min_up + 1)
            return sum(start_var[i] for i in range(start_idx, t + 1)) <= on_var[t]
        model.add_component(f'{prefix}_min_up', pyo.Constraint(model.T, rule=min_up_rule))

    if min_down > 0:
        def min_down_rule(m, t):
            start_idx = max(0, t - min_down + 1)
            return sum(stop_var[i] for i in range(start_idx, t + 1)) <= 1 - on_var[t]
        model.add_component(f'{prefix}_min_down', pyo.Constraint(model.T, rule=min_down_rule))

def check_unit_feasibility(unit_label, demand_vec, params, solve_hours, solve_start):
    total_demand = float(np.sum(demand_vec) * params.dt)
    n_steps = len(demand_vec)

    storage_energy_available = max(0.0, params.soc_0 - params.soc_min)
    max_storage_throughput = params.sto_discharge_max * n_steps * params.dt
    usable_storage = min(storage_energy_available, max_storage_throughput)

    if unit_label == 'Heat Pump only':
        max_supply = params.hp_p_el_max * params.hp_cop * n_steps * params.dt + usable_storage

    elif unit_label == 'Condensing Boiler only':
        max_supply = params.boiler_q_max * n_steps * params.dt + usable_storage

    elif unit_label == 'CHP only':
        max_supply = params.chp_p_el_max * params.chp_th_per_el * n_steps * params.dt + usable_storage

    else:
        return True

    if total_demand > max_supply:
        return False

    return True

def run_rolling_dispatch(approach_name, unit_label, base_params, build_model_fn, solve_hours, notes_suffix=''):
    """Run the rolling horizon dispatch for a given (approach, unit) combination."""
    global RESULTS
    profile_parts = []
    total_cost_eur = 0.0
    state_overrides = {}
    terminations = []
    slack_log = []
    slack_df = pd.DataFrame()
    started_at = perf_counter()

    max_start = min(
        heat_demand_raw.index.max(),
        electricity_prices_raw.index.max()
    ) - pd.Timedelta(hours=solve_hours)

    max_days = int((max_start - EVALUATION_START).total_seconds() // (24 * 3600)) + 1

    for day_idx in range(max_days):
        solve_start = EVALUATION_START + pd.Timedelta(hours=24 * day_idx)
        window_end = solve_start + pd.Timedelta(hours=solve_hours - 1)
        if window_end > heat_demand_raw.index.max() or window_end > electricity_prices_raw.index.max():
            print(f"Skipping {solve_start} → window exceeds dataset")
            continue

        demand_vec, price_vec, timestamps = prepare_hourly_window(
            solve_start, solve_hours, heat_demand_raw, electricity_prices_raw,
    )

        param_values = {f: getattr(base_params, f) for f in base_params.__dataclass_fields__}
        param_values.update(state_overrides)
        param_values['n_hours'] = solve_hours
        run_params = ModelParams(**param_values)

        feasible = check_unit_feasibility(unit_label, demand_vec, run_params, solve_hours, solve_start)

        if not feasible:
            print(f"⚠️ {unit_label} infeasible → MPC uses slack")

        model = build_model_fn(run_params, demand_vec, price_vec)
        solver = pyo.SolverFactory('appsi_highs')
        solver.options['time_limit'] = 120
        result = solver.solve(model)
        
        total_slack = sum(pyo.value(model.heat_slack[t]) for t in model.T)
        if total_slack > 1e-6:
            print(f"   ⚠️ [{solve_start.date()}] Unserved heat: {total_slack:.2f} MWh_th")
        
        slack_log.append({
            "timestamp": solve_start,
            "unserved_heat_mwh": total_slack
        })

        term = str(result.solver.termination_condition)

        if term == 'infeasible':
            print(f"❌ {unit_label} infeasible at {solve_start.date()}")
            return 0.0, float('inf'), pd.DataFrame()
        termination_condition = str(result.solver.termination_condition)
        terminations.append(f"{solve_start.date()}={termination_condition}")
        if termination_condition == 'infeasible':
            print(f"❌ {unit_label} infeasible at {solve_start.date()}")
            return 0.0, float('inf'), pd.DataFrame()

        elif termination_condition in {'optimal', 'feasible', 'maxTimeLimit'}:
            pass

        else:
            print(f"⚠️ Unexpected termination: {termination_condition}")

        committed_profile = build_result_profile(model, demand_vec, price_vec, timestamps).iloc[:COMMIT_STEPS].copy()
        committed_profile['timestamp'] = pd.to_datetime(committed_profile['timestamp'], utc=True)
        profile_parts.append(committed_profile)
        total_cost_eur += compute_committed_cost(model, run_params, COMMIT_STEPS)
        state_overrides = extract_commit_state(model, run_params, COMMIT_STEPS)

    runtime_seconds = perf_counter() - started_at
    profile = pd.concat(profile_parts, ignore_index=True)
    notes = (
        f"Unit={unit_label}; start={EVALUATION_START}; days={EVALUATION_DAYS}; "
        f"solve={solve_hours}h; commit={COMMIT_HOURS}h"
    )

    if notes_suffix:
        notes = f"{notes}; {notes_suffix}"

    notes = f"{notes}; terminations: {' | '.join(terminations)}"

    key = f"{approach_name} | {unit_label}"
    RESULTS = RESULTS.loc[~(
        (RESULTS['approach_name'] == approach_name) &
        (RESULTS['unit'] == unit_label)
    )].reset_index(drop=True)

    RESULTS.loc[len(RESULTS)] = {
        'approach_name': approach_name,
        'unit': unit_label,
        'runtime_seconds': runtime_seconds,
        'notes': notes,
        'total_cost_eur': total_cost_eur,
        'profile': profile,
        'slack_profile': slack_df,
    }

    print(f"  [{key}] done in {runtime_seconds:.1f}s | total cost: {total_cost_eur:.2f} EUR")

    slack_df = pd.DataFrame(slack_log)

    if len(slack_df) > 0:
        TOL = 1e-6
        total_slack = slack_df["unserved_heat_mwh"].clip(lower=0).sum()
        total_slack = total_slack if total_slack > TOL else 0.0
        #for slack price use effective gas cost to keep chp cost realistic
        slack_cost = total_slack * 47.06

        print("\n================ SLACK SUMMARY ================")
        print(f"Unit: {unit_label}")
        print(f"Total slack energy: {total_slack:.2f} MWh_th")
        print(f"Estimated cost: {slack_cost:.2f} €")

        print("\nProblem days:")

        TOL = 1e-6
        clean_slack = slack_df[slack_df["unserved_heat_mwh"] > TOL]

        if len(clean_slack) > 0:
            print(clean_slack.sort_values("unserved_heat_mwh", ascending=False))
        else:
            print("\nNo meaningful slack required (only numerical noise).")

        return runtime_seconds, total_cost_eur, profile

---
## Standard 24h MILP, single-unit variants

**Approach name:** Standard MILP (24h solve, 24h commit) — Heat Pump only / Boiler only / CHP only

**Main idea:** Identical to the full-system Approach, but with only one generation unit active at a time. All other unit dispatch variables are fixed to zero. The thermal storage remains active to allow feasibility.

**Difference to canonical formulation:** Two of the three production units are disabled per run (their `_on` binary and continuous output variables are present in the model but bounded to zero via additional constraints). This isolates the marginal contribution of each unit.

**Interpretation against evaluation baseline:** Provides a lower bound on the benefit of combining assets. A high single-unit cost relative to the full-system cost reveals where multi-asset dispatch creates value.

**Expectation:**
- **Heat Pump only**: Cheapest when electricity prices are low; will struggle during expensive peak hours, forcing heavy storage use or infeasibility near capacity limits.
- **Boiler only**: Cost driven purely by gas price; no electricity market exposure; highest or lowest cost depending on electricity vs. gas spread.
- **CHP only**: Revenue from electricity sales partially offsets fuel cost; minimum up/down constraints cause cycling inefficiency when running alone.

In [43]:
# =============================================================================
# Standard 24h MILP  |  Single-unit variants
# =============================================================================

def _build_single_unit_model_24h(p, demand_th, price_el, *, enable_hp, enable_boiler, enable_chp):
    """Core 24h MILP model; individual units can be switched on/off."""
    if len(demand_th) != p.n_steps or len(price_el) != p.n_steps:
        raise ValueError('Inputs must have length p.n_steps')

    m = pyo.ConcreteModel('SingleUnit24h')
    m.T = pyo.RangeSet(0, p.n_steps - 1)
    m.demand_th = pyo.Param(m.T, initialize={t: float(demand_th[t]) for t in range(p.n_steps)})
    m.price_el  = pyo.Param(m.T, initialize={t: float(price_el[t])  for t in range(p.n_steps)})

    # --- Binary variables ---
    m.hp_on        = pyo.Var(m.T, domain=pyo.Binary)
    m.boiler_on    = pyo.Var(m.T, domain=pyo.Binary)
    m.chp_on       = pyo.Var(m.T, domain=pyo.Binary)
    m.boiler_start = pyo.Var(m.T, domain=pyo.Binary)
    m.boiler_stop  = pyo.Var(m.T, domain=pyo.Binary)
    m.chp_start    = pyo.Var(m.T, domain=pyo.Binary)
    m.chp_stop     = pyo.Var(m.T, domain=pyo.Binary)
    m.sto_mode_charge = pyo.Var(m.T, domain=pyo.Binary)

    # --- Continuous variables ---
    m.hp_el_in        = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.hp_th_out       = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.boiler_th_out   = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.boiler_fuel_in  = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.chp_el_out      = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.chp_th_out      = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.chp_fuel_in     = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.sto_th_charge    = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.sto_th_discharge = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.sto_th_soc       = pyo.Var(m.T, domain=pyo.NonNegativeReals)

    #testvariable
    m.heat_slack = pyo.Var(m.T, domain=pyo.NonNegativeReals)

    # --- Disable unused units (fix outputs to zero) ---
    if not enable_hp:
        m.hp_disabled = pyo.Constraint(m.T, rule=lambda mm, t: mm.hp_on[t] == 0)
    if not enable_boiler:
        m.boiler_disabled = pyo.Constraint(m.T, rule=lambda mm, t: mm.boiler_on[t] == 0)
    if not enable_chp:
        m.chp_disabled = pyo.Constraint(m.T, rule=lambda mm, t: mm.chp_on[t] == 0)

    # --- Heat balance ---
    m.heat_balance = pyo.Constraint(
        m.T,
        rule=lambda mm, t: (
            mm.hp_th_out[t] + mm.boiler_th_out[t] + mm.chp_th_out[t]
            + mm.sto_th_discharge[t] - mm.sto_th_charge[t]
            + mm.heat_slack[t] == mm.demand_th[t]
        ),
    )

    # --- Storage ---
    m.sto_soc_upper = pyo.Constraint(m.T, rule=lambda mm, t: mm.sto_th_soc[t] <= p.sto_capacity)
    m.sto_soc_lower = pyo.Constraint(m.T, rule=lambda mm, t: mm.sto_th_soc[t] >= p.soc_min)
    m.sto_charge_limit    = pyo.Constraint(m.T, rule=lambda mm, t: mm.sto_th_charge[t]    <= p.sto_charge_max    * mm.sto_mode_charge[t])
    m.sto_discharge_limit = pyo.Constraint(m.T, rule=lambda mm, t: mm.sto_th_discharge[t] <= p.sto_discharge_max * (1 - mm.sto_mode_charge[t]))

    def soc_rule(mm, t):
        prev = p.soc_0 if t == 0 else mm.sto_th_soc[t - 1]
        return mm.sto_th_soc[t] == prev + p.dt * mm.sto_th_charge[t] - p.dt * mm.sto_th_discharge[t] - p.sto_loss
    m.sto_soc_dynamics = pyo.Constraint(m.T, rule=soc_rule)

    # --- Heat pump ---
    m.hp_min = pyo.Constraint(m.T, rule=lambda mm, t: p.hp_p_el_min * mm.hp_on[t] <= mm.hp_el_in[t])
    m.hp_max = pyo.Constraint(m.T, rule=lambda mm, t: mm.hp_el_in[t] <= p.hp_p_el_max * mm.hp_on[t])
    m.hp_cop = pyo.Constraint(m.T, rule=lambda mm, t: mm.hp_th_out[t] == p.hp_cop * mm.hp_el_in[t])

    # --- Boiler ---
    m.boiler_min = pyo.Constraint(m.T, rule=lambda mm, t: p.boiler_q_min * mm.boiler_on[t] <= mm.boiler_th_out[t])
    m.boiler_max = pyo.Constraint(m.T, rule=lambda mm, t: mm.boiler_th_out[t] <= p.boiler_q_max * mm.boiler_on[t])
    m.boiler_eta = pyo.Constraint(m.T, rule=lambda mm, t: mm.boiler_fuel_in[t] * p.boiler_eff == mm.boiler_th_out[t])
    m.boiler_start_logic = pyo.Constraint(m.T, rule=lambda mm, t: mm.boiler_start[t] >= mm.boiler_on[t] - (p.boiler_on_0 if t == 0 else mm.boiler_on[t - 1]))
    m.boiler_stop_logic  = pyo.Constraint(m.T, rule=lambda mm, t: mm.boiler_stop[t]  >= (p.boiler_on_0 if t == 0 else mm.boiler_on[t - 1]) - mm.boiler_on[t])
    add_min_up_down_constraints(m, m.boiler_on, m.boiler_start, m.boiler_stop, p.boiler_min_up, p.boiler_min_down, 'boiler')
    # --- CHP prefers charging storage ---
    def chp_storage_link(m, t):
        return m.sto_th_charge[t] <= p.chp_q_th_max
    m.chp_storage_link = pyo.Constraint(m.T, rule=chp_storage_link)

    # --- CHP ---
    m.chp_min  = pyo.Constraint(m.T, rule=lambda mm, t: p.chp_p_el_min * mm.chp_on[t] <= mm.chp_el_out[t])
    m.chp_max  = pyo.Constraint(m.T, rule=lambda mm, t: mm.chp_el_out[t] <= p.chp_p_el_max * mm.chp_on[t])
    m.chp_heat = pyo.Constraint(m.T, rule=lambda mm, t: mm.chp_th_out[t] == p.chp_th_per_el * mm.chp_el_out[t])
    m.chp_eta  = pyo.Constraint(m.T, rule=lambda mm, t: mm.chp_fuel_in[t] * p.chp_eta_el == mm.chp_el_out[t])
    m.chp_start_logic = pyo.Constraint(m.T, rule=lambda mm, t: mm.chp_start[t] >= mm.chp_on[t] - (p.chp_on_0 if t == 0 else mm.chp_on[t - 1]))
    m.chp_stop_logic  = pyo.Constraint(m.T, rule=lambda mm, t: mm.chp_stop[t]  >= (p.chp_on_0 if t == 0 else mm.chp_on[t - 1]) - mm.chp_on[t])
    add_min_up_down_constraints(m, m.chp_on, m.chp_start, m.chp_stop, p.chp_min_up, p.chp_min_down, 'chp')

    # --- Objective ---
    m.objective = pyo.Objective(
        expr=sum(
            m.hp_el_in[t] * m.price_el[t] * p.dt
            + m.boiler_fuel_in[t] * p.fuel_cost * p.dt
            + m.chp_fuel_in[t] * p.fuel_cost * p.dt
            + p.chp_startup_cost * m.chp_start[t]
            - m.chp_el_out[t] * m.price_el[t] * p.dt
            + p.slack_penalty * m.heat_slack[t] * p.dt
            
            for t in m.T
        ),
        sense=pyo.minimize,
    )
    return m


# Three build-function wrappers — one per unit
build_hp_only_24h     = lambda p, d, e: _build_single_unit_model_24h(p, d, e, enable_hp=True,  enable_boiler=False, enable_chp=False)
build_boiler_only_24h = lambda p, d, e: _build_single_unit_model_24h(p, d, e, enable_hp=False, enable_boiler=True,  enable_chp=False)
build_chp_only_24h    = lambda p, d, e: _build_single_unit_model_24h(p, d, e, enable_hp=False, enable_boiler=False, enable_chp=True)

APPROACH_1_NAME = 'Standard MILP (24h)'

print(f"Running {APPROACH_1_NAME} — Heat Pump only ...")
run_rolling_dispatch(APPROACH_1_NAME, 'Heat Pump only',       params, build_hp_only_24h,     solve_hours=24)

print(f"Running {APPROACH_1_NAME} — Condensing Boiler only ...")
run_rolling_dispatch(APPROACH_1_NAME, 'Condensing Boiler only', params, build_boiler_only_24h, solve_hours=24)

print(f"Running {APPROACH_1_NAME} — CHP only ...")
run_rolling_dispatch(APPROACH_1_NAME, 'CHP only',             params, build_chp_only_24h,    solve_hours=24)

print("\n24h Approach results:")
print(RESULTS[['approach_name', 'unit', 'runtime_seconds', 'total_cost_eur']].to_string(index=False))

Running Standard MILP (24h) — Heat Pump only ...
  [Standard MILP (24h) | Heat Pump only] done in 22.8s | total cost: 64307.13 EUR

================ SLACK SUMMARY ================
Unit: Heat Pump only
Total slack energy: 0.00 MWh_th
Estimated cost: 0.00 €

Problem days:

No meaningful slack required (only numerical noise).
Running Standard MILP (24h) — Condensing Boiler only ...
  [Standard MILP (24h) | Condensing Boiler only] done in 37.8s | total cost: 215374.10 EUR

================ SLACK SUMMARY ================
Unit: Condensing Boiler only
Total slack energy: 0.00 MWh_th
Estimated cost: 0.00 €

Problem days:

No meaningful slack required (only numerical noise).
Running Standard MILP (24h) — CHP only ...
   ⚠️ [2024-03-04] Unserved heat: 1.16 MWh_th
   ⚠️ [2024-03-05] Unserved heat: 12.39 MWh_th
⚠️ CHP only infeasible → MPC uses slack
   ⚠️ [2024-03-06] Unserved heat: 82.07 MWh_th
⚠️ CHP only infeasible → MPC uses slack
   ⚠️ [2024-03-07] Unserved heat: 68.31 MWh_th
   ⚠️ [2024-03-

---
##  MPC 35h solve / 24h commit, single-unit variants

**Approach name:** MPC prototype (35h solve, 24h commit) — Heat Pump only / Boiler only / CHP only

**Main idea:** Same as the full-system Approach — solve over a 35h horizon but commit only the first 24h — applied here to each generation unit in isolation. All other units are disabled as in the Approach 24h variants.

**Difference to canonical formulation:** Longer look-ahead (35h) without a terminal storage penalty; only one production asset active per run.

**Interpretation against evaluation baseline:**
- Compared to Approach 24h single-unit variants: shows whether the additional look-ahead helps for each asset individually.
- Compared to the full-system Approach 35h: reveals whether multi-asset cooperation or look-ahead horizon is the dominant cost driver.

**Expectation:**
- **Heat Pump only**: Larger look-ahead allows pre-charging storage when electricity is cheap, reducing total electricity cost.
- **Boiler only**: Marginal or no improvement — boiler cost is purely gas-driven, which is time-invariant; longer horizon adds little value.
- **CHP only**: Look-ahead may improve CHP scheduling around electricity price peaks but constrained by minimum up/down times.

In [44]:
# =============================================================================
#  MPC 35h solve / 24h commit  |  Single-unit variants
# =============================================================================

MPC_HORIZON_HOURS = 35


def _build_single_unit_model_35h(p, demand_th, price_el, *, enable_hp, enable_boiler, enable_chp):
    """35h MPC model; individual units can be switched on/off. No terminal penalty."""
    if len(demand_th) != p.n_steps or len(price_el) != p.n_steps:
        raise ValueError('Inputs must have length p.n_steps')

    m = pyo.ConcreteModel('SingleUnit35h')
    m.T = pyo.RangeSet(0, p.n_steps - 1)
    m.demand_th = pyo.Param(m.T, initialize={t: float(demand_th[t]) for t in range(p.n_steps)})
    m.price_el  = pyo.Param(m.T, initialize={t: float(price_el[t])  for t in range(p.n_steps)})

    # --- Binary variables ---
    m.hp_on        = pyo.Var(m.T, domain=pyo.Binary)
    m.boiler_on    = pyo.Var(m.T, domain=pyo.Binary)
    m.chp_on       = pyo.Var(m.T, domain=pyo.Binary)
    m.boiler_start = pyo.Var(m.T, domain=pyo.Binary)
    m.boiler_stop  = pyo.Var(m.T, domain=pyo.Binary)
    m.chp_start    = pyo.Var(m.T, domain=pyo.Binary)
    m.chp_stop     = pyo.Var(m.T, domain=pyo.Binary)
    m.sto_mode_charge = pyo.Var(m.T, domain=pyo.Binary)

    # --- Continuous variables ---
    m.hp_el_in        = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.hp_th_out       = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.boiler_th_out   = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.boiler_fuel_in  = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.chp_el_out      = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.chp_th_out      = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.chp_fuel_in     = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.sto_th_charge    = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.sto_th_discharge = pyo.Var(m.T, domain=pyo.NonNegativeReals)
    m.sto_th_soc       = pyo.Var(m.T, domain=pyo.NonNegativeReals)

    #testvariable
    m.heat_slack = pyo.Var(m.T, domain=pyo.NonNegativeReals)

    # --- Disable unused units ---
    if not enable_hp:
        m.hp_disabled = pyo.Constraint(m.T, rule=lambda mm, t: mm.hp_on[t] == 0)
    if not enable_boiler:
        m.boiler_disabled = pyo.Constraint(m.T, rule=lambda mm, t: mm.boiler_on[t] == 0)
    if not enable_chp:
        m.chp_disabled = pyo.Constraint(m.T, rule=lambda mm, t: mm.chp_on[t] == 0)

    # --- Heat balance ---
    m.heat_balance = pyo.Constraint(
        m.T,
        rule=lambda mm, t: (
            mm.hp_th_out[t] + mm.boiler_th_out[t] + mm.chp_th_out[t]
            + mm.sto_th_discharge[t] - mm.sto_th_charge[t]
            + mm.heat_slack[t] == mm.demand_th[t]
        ),
    )

    # --- Storage ---
    m.sto_soc_upper = pyo.Constraint(m.T, rule=lambda mm, t: mm.sto_th_soc[t] <= p.sto_capacity)
    m.sto_soc_lower = pyo.Constraint(m.T, rule=lambda mm, t: mm.sto_th_soc[t] >= p.soc_min)
    m.sto_charge_limit    = pyo.Constraint(m.T, rule=lambda mm, t: mm.sto_th_charge[t]    <= p.sto_charge_max    * mm.sto_mode_charge[t])
    m.sto_discharge_limit = pyo.Constraint(m.T, rule=lambda mm, t: mm.sto_th_discharge[t] <= p.sto_discharge_max * (1 - mm.sto_mode_charge[t]))

    def soc_rule(mm, t):
        prev = p.soc_0 if t == 0 else mm.sto_th_soc[t - 1]
        return mm.sto_th_soc[t] == prev + p.dt * mm.sto_th_charge[t] - p.dt * mm.sto_th_discharge[t] - p.sto_loss
    m.sto_soc_dynamics = pyo.Constraint(m.T, rule=soc_rule)

    # --- Heat pump ---
    m.hp_min = pyo.Constraint(m.T, rule=lambda mm, t: p.hp_p_el_min * mm.hp_on[t] <= mm.hp_el_in[t])
    m.hp_max = pyo.Constraint(m.T, rule=lambda mm, t: mm.hp_el_in[t] <= p.hp_p_el_max * mm.hp_on[t])
    m.hp_cop = pyo.Constraint(m.T, rule=lambda mm, t: mm.hp_th_out[t] == p.hp_cop * mm.hp_el_in[t])

    # --- Boiler ---
    m.boiler_min = pyo.Constraint(m.T, rule=lambda mm, t: p.boiler_q_min * mm.boiler_on[t] <= mm.boiler_th_out[t])
    m.boiler_max = pyo.Constraint(m.T, rule=lambda mm, t: mm.boiler_th_out[t] <= p.boiler_q_max * mm.boiler_on[t])
    m.boiler_eta = pyo.Constraint(m.T, rule=lambda mm, t: mm.boiler_fuel_in[t] * p.boiler_eff == mm.boiler_th_out[t])
    m.boiler_start_logic = pyo.Constraint(m.T, rule=lambda mm, t: mm.boiler_start[t] >= mm.boiler_on[t] - (p.boiler_on_0 if t == 0 else mm.boiler_on[t - 1]))
    m.boiler_stop_logic  = pyo.Constraint(m.T, rule=lambda mm, t: mm.boiler_stop[t]  >= (p.boiler_on_0 if t == 0 else mm.boiler_on[t - 1]) - mm.boiler_on[t])
    add_min_up_down_constraints(m, m.boiler_on, m.boiler_start, m.boiler_stop, p.boiler_min_up, p.boiler_min_down, 'boiler_mpc')

    # --- CHP ---
    m.chp_min  = pyo.Constraint(m.T, rule=lambda mm, t: p.chp_p_el_min * mm.chp_on[t] <= mm.chp_el_out[t])
    m.chp_max  = pyo.Constraint(m.T, rule=lambda mm, t: mm.chp_el_out[t] <= p.chp_p_el_max * mm.chp_on[t])
    m.chp_heat = pyo.Constraint(m.T, rule=lambda mm, t: mm.chp_th_out[t] == p.chp_th_per_el * mm.chp_el_out[t])
    m.chp_eta  = pyo.Constraint(m.T, rule=lambda mm, t: mm.chp_fuel_in[t] * p.chp_eta_el == mm.chp_el_out[t])
    m.chp_start_logic = pyo.Constraint(m.T, rule=lambda mm, t: mm.chp_start[t] >= mm.chp_on[t] - (p.chp_on_0 if t == 0 else mm.chp_on[t - 1]))
    m.chp_stop_logic  = pyo.Constraint(m.T, rule=lambda mm, t: mm.chp_stop[t]  >= (p.chp_on_0 if t == 0 else mm.chp_on[t - 1]) - mm.chp_on[t])
    add_min_up_down_constraints(m, m.chp_on, m.chp_start, m.chp_stop, p.chp_min_up, p.chp_min_down, 'chp_mpc')
    # --- CHP prefers charging storage ---
    def chp_storage_link(m, t):
        return m.sto_th_charge[t] <= p.chp_q_th_max
    m.chp_storage_link = pyo.Constraint(m.T, rule=chp_storage_link)

    # --- Objective (full horizon, no terminal penalty) ---
    m.objective = pyo.Objective(
        expr=sum(
            m.hp_el_in[t] * m.price_el[t] * p.dt
            + m.boiler_fuel_in[t] * p.fuel_cost * p.dt
            + m.chp_fuel_in[t] * p.fuel_cost * p.dt
            + p.chp_startup_cost * m.chp_start[t]
            - m.chp_el_out[t] * m.price_el[t] * p.dt
            + p.slack_penalty * m.heat_slack[t] * p.dt

            for t in m.T
        ),
        sense=pyo.minimize,
    )
    return m


# Three build-function wrappers — one per unit
build_hp_only_35h     = lambda p, d, e: _build_single_unit_model_35h(p, d, e, enable_hp=True,  enable_boiler=False, enable_chp=False)
build_boiler_only_35h = lambda p, d, e: _build_single_unit_model_35h(p, d, e, enable_hp=False, enable_boiler=True,  enable_chp=False)
build_chp_only_35h    = lambda p, d, e: _build_single_unit_model_35h(p, d, e, enable_hp=False, enable_boiler=False, enable_chp=True)

APPROACH_3_NAME = 'MPC (35h solve, 24h commit)'

print(f"Running {APPROACH_3_NAME} — Heat Pump only ...")
run_rolling_dispatch(APPROACH_3_NAME, 'Heat Pump only',        params, build_hp_only_35h,     solve_hours=MPC_HORIZON_HOURS, notes_suffix='35h look-ahead, no terminal penalty')

print(f"Running {APPROACH_3_NAME} — Condensing Boiler only ...")
run_rolling_dispatch(APPROACH_3_NAME, 'Condensing Boiler only', params, build_boiler_only_35h, solve_hours=MPC_HORIZON_HOURS, notes_suffix='35h look-ahead, no terminal penalty')

print(f"Running {APPROACH_3_NAME} — CHP only ...")
run_rolling_dispatch(APPROACH_3_NAME, 'CHP only',              params, build_chp_only_35h,    solve_hours=MPC_HORIZON_HOURS, notes_suffix='35h look-ahead, no terminal penalty')

print("\nAll results so far:")
print(RESULTS[['approach_name', 'unit', 'runtime_seconds', 'total_cost_eur']].to_string(index=False))

Running MPC (35h solve, 24h commit) — Heat Pump only ...
  [MPC (35h solve, 24h commit) | Heat Pump only] done in 32.5s | total cost: 61036.03 EUR

================ SLACK SUMMARY ================
Unit: Heat Pump only
Total slack energy: 0.00 MWh_th
Estimated cost: 0.00 €

Problem days:

No meaningful slack required (only numerical noise).
Running MPC (35h solve, 24h commit) — Condensing Boiler only ...
  [MPC (35h solve, 24h commit) | Condensing Boiler only] done in 61.8s | total cost: 215408.61 EUR

================ SLACK SUMMARY ================
Unit: Condensing Boiler only
Total slack energy: 0.00 MWh_th
Estimated cost: 0.00 €

Problem days:

No meaningful slack required (only numerical noise).
Running MPC (35h solve, 24h commit) — CHP only ...
⚠️ CHP only infeasible → MPC uses slack
   ⚠️ [2024-03-05] Unserved heat: 27.20 MWh_th
⚠️ CHP only infeasible → MPC uses slack
   ⚠️ [2024-03-06] Unserved heat: 104.03 MWh_th
⚠️ CHP only infeasible → MPC uses slack
   ⚠️ [2024-03-07] Unserved

---
## Evaluation and Comparison

Summary table and dispatch plots for all 6 configurations (3 units × 2 approaches).

In [45]:
# ---------------------------------------------------------------------------
# Summary table
# ---------------------------------------------------------------------------
if RESULTS.empty:
    print("No results registered yet.")
else:
    summary = RESULTS[['approach_name', 'unit', 'runtime_seconds', 'total_cost_eur']].copy()
    # Compute cost saving of Approach 3 vs Approach 1 for each unit
    pivot = summary.pivot(index='unit', columns='approach_name', values='total_cost_eur')
    a1_col = [c for c in pivot.columns if 'Standard' in c]
    a3_col = [c for c in pivot.columns if 'MPC' in c]
    if a1_col and a3_col:
        pivot['saving_eur'] = pivot[a1_col[0]] - pivot[a3_col[0]]
        pivot['saving_pct'] = (pivot['saving_eur'] / pivot[a1_col[0]] * 100).round(2)
    print("=== Cost summary ===")
    print(summary.to_string(index=False))
    print()
    print("=== Approach 35h vs Approach 24h savings per unit ===")
    print(pivot.to_string())

=== Cost summary ===
              approach_name                   unit  runtime_seconds  total_cost_eur
        Standard MILP (24h)         Heat Pump only        22.830378    64307.134278
        Standard MILP (24h) Condensing Boiler only        37.757770   215374.098093
        Standard MILP (24h)               CHP only        27.489502   198331.320159
MPC (35h solve, 24h commit)         Heat Pump only        32.533836    61036.027152
MPC (35h solve, 24h commit) Condensing Boiler only        61.752780   215408.608375
MPC (35h solve, 24h commit)               CHP only        37.123792   182627.622124

=== Approach 35h vs Approach 24h savings per unit ===
approach_name           MPC (35h solve, 24h commit)  Standard MILP (24h)    saving_eur  saving_pct
unit                                                                                              
CHP only                              182627.622124        198331.320159  15703.698035        7.92
Condensing Boiler only                2